# Day 05：手写 MLP —— 用代码解决 XOR 问题

> ☀️ 第二周 · 破局与复兴 · 第 5 天

前四天我们逐个击破了神经网络的核心组件：

| 天 | 主题 | 解决了什么问题 |
|----|------|---------------|
| Day01 | 隐藏层 | 线性不可分 → 线性可分（空间变换） |
| Day02 | 激活函数 | 让网络有非线性能力（Sigmoid/ReLU） |
| Day03 | 损失函数与梯度下降 | 衡量错误 + 知道怎么改 |
| Day04 | 反向传播 | 把误差分摊给每个权重 |

今天，把这四块拼成一个完整的神经网络——**MLP（多层感知机）**。

而且，我们要告别「手写每一行梯度」的痛苦，拥抱 PyTorch 的 **autograd**（自动求导）——只写前向传播，梯度自动算！

**今天的任务**：用 PyTorch 的 autograd + nn.Module 实现完整的 MLP 训练流程，彻底解决 XOR 问题。

---
## 1. 历史背景：从手工作坊到自动化流水线

### 手写反向传播的痛苦

前两天我们手写了反向传播的每一步：
- 手动计算 `sigmoid_grad(a) = a * (1-a)`
- 手动计算 `dL_dz2 = dL_da2 * sigmoid_grad(a2)`
- 手动计算 `dL_dW2 = a1.T @ dL_dz2`
- 手动把梯度从输出层传回隐藏层

对于两层网络，这已经很繁琐了。想象一下，如果网络有 50 层、100 层——手写梯度根本不可能。

### 自动求导的革命

2012 年前后，随着深度学习的复兴，各大框架开始内置**自动求导（Automatic Differentiation）**功能：

| 年份 | 框架 | 特点 |
|------|------|------|
| 2007 | Torch (Lua) | 手动写梯度，研究者专用 |
| 2012 | Caffe | C++ 实现，CNN 专用 |
| 2015 | TensorFlow | Google 出品，计算图 + 自动求导 |
| 2016 | PyTorch | Facebook 出品，动态图 + Python 风格，迅速成为学术界首选 |

PyTorch 的核心理念是：**你只管写前向传播（怎么算），梯度自动帮你算出来**。

这就像从「手洗衣服」进化到「洗衣机」——你只需要把衣服放进去、按下按钮，剩下的交给机器。

---
## 2. 生活隐喻：自动炒菜机

### 手动烹饪 vs 自动炒菜机

想象你在做一道红烧肉：

**手写反向传播**的做法（手烹饪）：
1. 切肉（准备数据）
2. 炒糖色（第一层计算）
3. 加酱油（第二层计算）
4. 尝味道，觉得太咸了（计算损失）
5. **自己分析**：是酱油放多了？还是盐放多了？每一味调料对咸度的贡献是多少？（手动算梯度）
6. 根据分析结果调整调料（更新权重）

**autograd**的做法（用自动炒菜机）：
1. 把食材放进去（准备数据）
2. 选择菜谱（定义模型结构）
3. 按下「开始」（前向传播）
4. 机器自动尝味道、自动分析、自动调整（autograd 自动算梯度 + optimizer 自动更新）
5. 你只需要告诉它「目标味道」（损失函数）

autograd 就是一个**自动调味系统**——你不需要知道每味料的精确影响，机器帮你算。

### 关键简化

| 手动模式 | 自动模式 | 对应操作 |
|---------|---------|----------|
| 手算 sigmoid 导数 | 自动 | `loss.backward()` |
| 手算链式法则 | 自动 | `loss.backward()` |
| 手算 dL/dW | 自动 | `W.grad` 自动生成 |
| 手动更新 W | optimizer 帮你做 | `optimizer.step()` |
| 手动清零梯度 | 需要手动调用 | `optimizer.zero_grad()` |

---
## 3. 自动求导：PyTorch 的核心魔法

### requires_grad=True

PyTorch 的自动求导只需要一个开关：`requires_grad=True`。

当你创建一个张量时，加上这个参数，PyTorch 就会**追踪**对它做过的所有运算。

当调用 `loss.backward()` 时，PyTorch 会沿着计算图**反向**遍历，自动算出每个 `requires_grad=True` 的张量的梯度。

整个过程只需要三步：
1. **标记**：`requires_grad=True` 告诉 PyTorch「这个参数要学」
2. **前向**：正常写计算逻辑，PyTorch 在幕后记录计算图
3. **反向**：`loss.backward()` 自动算出所有梯度

用代码验证这个魔法：

In [ ]:
import torch

# ===== 第 1 步：标记要学习的参数 =====
# requires_grad=True 告诉 PyTorch：「追踪对这个张量的所有运算」
x = torch.tensor([[1.0, 2.0]], requires_grad=True)  # 输入 [1, 2]
W = torch.tensor([[3.0], [4.0]], requires_grad=True)  # 权重 [[3], [4]]
b = torch.tensor([[5.0]], requires_grad=True)  # 偏置 [[5]]

# ===== 第 2 步：前向传播（正常写计算逻辑）=====
# PyTorch 在幕后记录每一步运算，构建「计算图」
z = torch.matmul(x, W) + b  # z = 1*3 + 2*4 + 5 = 16
y = torch.sigmoid(z)  # y = σ(16) ≈ 1.0
loss = (y - 0.5) ** 2  # 假设目标是 0.5，计算 MSE 损失

print("前向传播结果：")
print(f"  z = x·W + b = 1×3 + 2×4 + 5 = {z.item():.1f}")
print(f"  y = σ(z) = σ({z.item():.1f}) = {y.item():.4f}")
print(f"  loss = (y - 0.5)² = {loss.item():.4f}")

# ===== 第 3 步：反向传播（一行代码！）=====
# loss.backward() 自动计算所有 requires_grad=True 的张量的梯度
loss.backward()

print("\n反向传播（自动计算的梯度）：")
print(f"  ∂loss/∂W = {W.grad.flatten().tolist()}")  # 权重梯度
print(f"  ∂loss/∂b = {b.grad.item()}")  # 偏置梯度
print(f"  ∂loss/∂x = {x.grad.flatten().tolist()}")  # 输入梯度（通常不需要）

print("\n对比手写反向传播：")
print("  手动：需要自己推导 sigmoid 导数、链式法则、矩阵乘法...")
print("  自动：loss.backward() 一行搞定！")
print("  PyTorch 的 autograd 就是你的「自动调味系统」。")

---
## 4. 从手写到 nn.Module：封装的力量

昨天我们手写了 `TwoLayerMLP` 类，里面每一层的权重、偏置、前向传播、反向传播都要自己管理。

PyTorch 提供了 `nn.Module`——一个基类，帮你封装了：
- **参数管理**：自动收集所有权重和偏置
- **自动求导**：前向传播时自动构建计算图
- **设备管理**：一行代码把模型搬到 GPU
- **保存/加载**：一键保存和恢复模型

用 `nn.Module` 重写 MLP，核心变化：

| 手写模式 | nn.Module 模式 |
|---------|---------------|
| `self.W1 = torch.randn(2, 4)` | `self.linear1 = nn.Linear(2, 4)` |
| `z1 = x @ W1 + b1` | `z1 = self.linear1(x)` |
| 手动算梯度 | `loss.backward()` 自动算 |
| 手动更新 `W -= lr * grad` | `optimizer.step()` 自动更新 |

### 用 nn.Module 实现 MLP

一个完整的 MLP 结构：

```
输入 (2维)
  │
  ↓
[Linear(2→8)] ← 隐藏层：2个输入 → 8个隐藏神经元
  │
  ↓
[Sigmoid()] ← 非线性激活
  │
  ↓
[Linear(8→1)] ← 输出层：8个隐藏 → 1个输出
  │
  ↓
[Sigmoid()] ← 输出激活（二分类输出概率）
  │
  ↓
输出 (1维)
```

用代码实现：

In [ ]:
import torch

class MLP(torch.nn.Module):
    """用 PyTorch nn.Module 实现的多层感知机

    结构：输入(2) → 隐藏层(8, Sigmoid) → 输出(1, Sigmoid)
    """

    def __init__(self, input_size, hidden_size, output_size):
        """初始化网络层

        参数:
            input_size: 输入维度（XOR 问题为 2）
            hidden_size: 隐藏层神经元数量（建议 4~8）
            output_size: 输出维度（二分类为 1）
        """
        super().__init__()  # 调用父类 nn.Module 的初始化

        # 隐藏层：线性变换 + 激活函数
        self.linear1 = torch.nn.Linear(input_size, hidden_size)  # [2, 8]
        self.activation1 = torch.nn.Sigmoid()  # Sigmoid 激活

        # 输出层：线性变换 + 激活函数
        self.linear2 = torch.nn.Linear(hidden_size, output_size)  # [8, 1]
        self.activation2 = torch.nn.Sigmoid()  # Sigmoid 输出概率

    def forward(self, x):
        """前向传播：x → 隐藏层 → 输出

        参数:
            x: 输入张量 [batch_size, input_size]
        返回:
            输出概率 [batch_size, 1]，值在 (0, 1) 之间
        """
        # 隐藏层：线性变换 + Sigmoid 激活
        x = self.linear1(x)  # [batch, 2] → [batch, 8]
        x = self.activation1(x)  # Sigmoid 激活

        # 输出层：线性变换 + Sigmoid 激活
        x = self.linear2(x)  # [batch, 8] → [batch, 1]
        x = self.activation2(x)  # 输出概率

        return x

# 创建模型
model = MLP(input_size=2, hidden_size=8, output_size=1)

# 查看模型结构
print("MLP 架构：")
print(model)

# 查看参数形状
print("\n模型参数：")
for name, param in model.named_parameters():
    print(f"  {name}: 形状={param.shape}")

---
## 5. 训练循环：五步走

PyTorch 的训练循环是固定的五步，和昨天手写的流程完全对应：

| 步骤 | 手写模式 | PyTorch 模式 |
|------|---------|-------------|
| 1. 前向传播 | `y_pred = model.forward(X)` | `y_pred = model(X)` |
| 2. 计算损失 | `loss = torch.mean((y_pred - y)**2)` | `loss = criterion(y_pred, y)` |
| 3. 清零梯度 | `W.grad.zero_()` 每个参数都要清 | `optimizer.zero_grad()` 一行清零 |
| 4. 反向传播 | 手动算每一层的梯度 | `loss.backward()` 自动算 |
| 5. 更新参数 | `W -= lr * W.grad` 每个参数都要更新 | `optimizer.step()` 一行更新 |

另外，还需要两个「辅助工具」：
- **优化器（Optimizer）**：管理参数更新的策略，比如 Adam
- **损失函数（Criterion）**：衡量预测和真实的差距，比如 MSE

用代码实现完整的 XOR 训练：

In [ ]:
import torch

# ===== MLP 类定义（和上面相同）=====
class MLP(torch.nn.Module):
    """用 PyTorch nn.Module 实现的多层感知机"""
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_size, hidden_size)  # 隐藏层
        self.activation1 = torch.nn.Sigmoid()  # 隐藏层激活
        self.linear2 = torch.nn.Linear(hidden_size, output_size)  # 输出层
        self.activation2 = torch.nn.Sigmoid()  # 输出层激活

    def forward(self, x):
        """前向传播"""
        x = self.activation1(self.linear1(x))  # 隐藏层：线性 + 激活
        x = self.activation2(self.linear2(x))  # 输出层：线性 + 激活
        return x

# ===== XOR 数据 =====
X_xor = torch.tensor([
    [0.0, 0.0],  # 两个都是 0
    [1.0, 0.0],  # 第一个是 1
    [0.0, 1.0],  # 第二个是 1
    [1.0, 1.0],  # 两个都是 1
], dtype=torch.float32)

# 标准 XOR 标签：(0,0)->0, (1,0)->1, (0,1)->1, (1,1)->0
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

# ===== 创建模型和辅助工具 =====
model = MLP(input_size=2, hidden_size=8, output_size=1)  # 2输入 → 8隐藏 → 1输出
optimizer = torch.optim.Adam(model.parameters(), lr=0.5)  # Adam 优化器
criterion = torch.nn.MSELoss()  # MSE 损失函数

# ===== 训练 =====
print("训练 XOR（PyTorch nn.Module + autograd）")
print("=" * 55)

losses = []  # 记录每轮损失，用于画图

for epoch in range(500):
    # 第 1 步：清零梯度（PyTorch 默认累加梯度，必须手动清零）
    optimizer.zero_grad()

    # 第 2 步：前向传播（计算预测值）
    y_pred = model(X_xor)  # [4, 1]

    # 第 3 步：计算损失（MSE）
    loss = criterion(y_pred, y_xor)
    losses.append(loss.item())  # 记录损失值

    # 第 4 步：反向传播（自动计算所有参数的梯度）
    loss.backward()

    # 第 5 步：更新参数（optimizer 自动用梯度更新所有参数）
    optimizer.step()

    # 每 100 轮打印一次损失
    if (epoch + 1) % 100 == 0:
        print(f"  第 {epoch+1:3d} 轮：损失 = {loss.item():.4f}")

print(f"\n训练完成！最终损失 = {losses[-1]:.4f}")

---
## 6. 验证实验

训练完了，验证模型是否真的学会了 XOR。

同时画出损失曲线和预测结果对比图：

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# ===== MLP 类定义 =====
class MLP(torch.nn.Module):
    """用 PyTorch nn.Module 实现的多层感知机"""
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_size, hidden_size)  # 隐藏层
        self.activation1 = torch.nn.Sigmoid()  # 隐藏层激活
        self.linear2 = torch.nn.Linear(hidden_size, output_size)  # 输出层
        self.activation2 = torch.nn.Sigmoid()  # 输出层激活

    def forward(self, x):
        """前向传播"""
        x = self.activation1(self.linear1(x))  # 隐藏层
        x = self.activation2(self.linear2(x))  # 输出层
        return x

# ===== XOR 数据和训练 =====
X_xor = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], dtype=torch.float32)
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

model = MLP(input_size=2, hidden_size=8, output_size=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.5)
criterion = torch.nn.MSELoss()

# 训练 500 轮
losses = []
for epoch in range(500):
    optimizer.zero_grad()  # 清零梯度
    y_pred = model(X_xor)  # 前向传播
    loss = criterion(y_pred, y_xor)  # 计算损失
    losses.append(loss.item())  # 记录损失
    loss.backward()  # 反向传播
    optimizer.step()  # 更新参数

# ===== 验证 =====
model.eval()  # 切换到评估模式（关闭 Dropout 等训练专用行为）
with torch.no_grad():  # 验证时不需要计算梯度
    y_pred = model(X_xor)  # 获取预测结果

# 画图：损失曲线 + 预测对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图：损失曲线
axes[0].plot(losses, 'b-', linewidth=1.5)  # 蓝色曲线
axes[0].set_xlabel('训练轮次')  # x 轴标签
axes[0].set_ylabel('损失 (MSE)')  # y 轴标签
axes[0].set_title('XOR 训练损失曲线')  # 标题
axes[0].grid(True, alpha=0.3)  # 网格

# 右图：预测结果对比
categories = ['(0,0)', '(1,0)', '(0,1)', '(1,1)']  # 四个输入
x_pos = np.arange(len(categories))  # x 位置
width = 0.35  # 柱子宽度

axes[1].bar(x_pos - width/2, y_xor.flatten().tolist(), width,
            label='真实值', color='steelblue', alpha=0.8)  # 真实值柱子
axes[1].bar(x_pos + width/2, y_pred.flatten().tolist(), width,
            label='预测值', color='coral', alpha=0.8)  # 预测值柱子
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)  # 决策阈值线
axes[1].set_xticks(x_pos)  # 设置 x 轴刻度
axes[1].set_xticklabels(categories)  # 设置 x 轴标签
axes[1].set_ylabel('输出')  # y 轴标签
axes[1].set_title('XOR 预测结果对比')  # 标题
axes[1].legend()  # 图例
axes[1].set_ylim(-0.1, 1.1)  # y 轴范围

plt.tight_layout()  # 自动调整布局
plt.savefig('day05_xor_result.png', dpi=150, bbox_inches='tight')  # 保存图片
plt.show()  # 显示图片

# 打印详细结果
print("\n详细预测结果：")
print("=" * 55)
for i in range(4):
    x_str = f"({int(X_xor[i,0].item())}, {int(X_xor[i,1].item())})"  # 输入
    true_val = int(y_xor[i].item())  # 真实标签
    pred_prob = y_pred[i].item()  # 预测概率
    pred_val = 1 if pred_prob > 0.5 else 0  # 预测值（大于 0.5 判为 1）
    status = "✓ 正确" if true_val == pred_val else "✗ 错误"  # 判断对错
    print(f"  {x_str}: 真实={true_val}, 预测={pred_val} (概率={pred_prob:.4f}) {status}")

---
## 7. 可视化决策边界

### 什么是决策边界？

决策边界是模型「判断 0 还是 1」的分界线。

对于单层感知机，决策边界永远是一条**直线**——这是它无法解决 XOR 的根本原因。

对于 MLP，隐藏层把数据变换到了新空间，决策边界变成了一条**曲线**——这就是非线性分类的力量。

用热力图可视化 MLP 学到的决策边界：

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# ===== MLP 类定义 =====
class MLP(torch.nn.Module):
    """用 PyTorch nn.Module 实现的多层感知机"""
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_size, hidden_size)
        self.activation1 = torch.nn.Sigmoid()
        self.linear2 = torch.nn.Linear(hidden_size, output_size)
        self.activation2 = torch.nn.Sigmoid()

    def forward(self, x):
        """前向传播"""
        x = self.activation1(self.linear1(x))
        x = self.activation2(self.linear2(x))
        return x

# ===== XOR 数据和训练 =====
X_xor = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], dtype=torch.float32)
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

model = MLP(input_size=2, hidden_size=8, output_size=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.5)
criterion = torch.nn.MSELoss()

# 训练 500 轮
for epoch in range(500):
    optimizer.zero_grad()
    y_pred = model(X_xor)
    loss = criterion(y_pred, y_xor)
    loss.backward()
    optimizer.step()

# ===== 创建网格（用于画决策边界）=====
x1_range = np.linspace(-0.5, 1.5, 100)  # X1 轴范围
x2_range = np.linspace(-0.5, 1.5, 100)  # X2 轴范围
X1, X2 = np.meshgrid(x1_range, x2_range)  # 生成网格

# 把网格点转成张量，送入模型预测
grid_points = torch.tensor(np.column_stack([X1.ravel(), X2.ravel()]), dtype=torch.float32)  # [10000, 2]

model.eval()  # 评估模式
with torch.no_grad():  # 不计算梯度
    y_grid = model(grid_points).numpy().reshape(X1.shape)  # 预测并 reshape 回网格形状

# ===== 画决策边界 =====
plt.figure(figsize=(10, 8))

# 热力图：颜色越深越接近 0，越浅越接近 1
contour = plt.contourf(X1, X2, y_grid, levels=20, cmap='RdYlGn', alpha=0.8)
plt.colorbar(contour, label='预测概率')  # 颜色条

# 决策边界线：预测概率 = 0.5 的等高线
plt.contour(X1, X2, y_grid, levels=[0.5], colors='black', linewidths=2)

# 画 XOR 的四个数据点
point_colors = ['green' if y == 1 else 'red' for y in y_xor.numpy()]  # 1=绿，0=红
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=point_colors, s=300, edgecolors='black', linewidth=2, zorder=5)

# 标注每个点的标签
for i in range(4):
    label = f'({int(X_xor[i,0].item())},{int(X_xor[i,1].item())})→{int(y_xor[i].item())}'
    plt.annotate(label, (X_xor[i, 0], X_xor[i, 1]),
                 textcoords="offset points", xytext=(15, 15), fontsize=11)

plt.xlabel('X1')  # x 轴标签
plt.ylabel('X2')  # y 轴标签
plt.title('MLP 学到的 XOR 决策边界\n(黑线 = 0.5 决策边界)')  # 标题
plt.grid(True, alpha=0.3)  # 网格

plt.tight_layout()  # 自动调整布局
plt.savefig('day05_decision_boundary.png', dpi=150, bbox_inches='tight')  # 保存图片
plt.show()  # 显示图片

print("看！MLP 学到了一条弯曲的决策边界（黑线）！")
print("红点 (0,0) 和 (1,1) 在一边，绿点 (1,0) 和 (0,1) 在另一边。")
print("这就是隐藏层的功劳——它把线性不可分的问题变换到了线性可分的空间。")

---
## 8. 对比：单层 vs 多层

最后，让我们直观对比单层感知机和 MLP 的区别。

| 对比项 | 单层感知机 | MLP（多层） |
|--------|-----------|-------------|
| 决策边界 | 永远是直线 | 可以是曲线 |
| 能否解决 XOR | 不能（数学证明） | 能（隐藏层的空间变换） |
| 损失曲线 | 收敛到某个值后不再下降 | 持续下降到接近 0 |

用代码对比两者的损失曲线和决策边界：

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# ===== 模型定义 =====
class MLP(torch.nn.Module):
    """多层感知机（有隐藏层）"""
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_size, hidden_size)  # 隐藏层
        self.activation1 = torch.nn.Sigmoid()  # 隐藏层激活
        self.linear2 = torch.nn.Linear(hidden_size, output_size)  # 输出层
        self.activation2 = torch.nn.Sigmoid()  # 输出层激活

    def forward(self, x):
        """前向传播"""
        x = self.activation1(self.linear1(x))  # 隐藏层
        x = self.activation2(self.linear2(x))  # 输出层
        return x

class SingleLayerPerceptron(torch.nn.Module):
    """单层感知机（无隐藏层）"""
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(2, 1)  # 直接 2→1
        self.activation = torch.nn.Sigmoid()  # Sigmoid 激活

    def forward(self, x):
        """前向传播"""
        return self.activation(self.linear(x))  # 线性 + 激活

# ===== XOR 数据 =====
X_xor = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], dtype=torch.float32)
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

# ===== 训练 MLP（有隐藏层）=====
model_mlp = MLP(input_size=2, hidden_size=8, output_size=1)  # 创建 MLP
optimizer_mlp = torch.optim.Adam(model_mlp.parameters(), lr=0.5)  # Adam 优化器
criterion = torch.nn.MSELoss()  # MSE 损失

losses_mlp = []  # MLP 损失记录
for epoch in range(500):
    optimizer_mlp.zero_grad()  # 清零梯度
    y_pred = model_mlp(X_xor)  # 前向传播
    loss = criterion(y_pred, y_xor)  # 计算损失
    losses_mlp.append(loss.item())  # 记录损失
    loss.backward()  # 反向传播
    optimizer_mlp.step()  # 更新参数

# ===== 训练单层感知机（无隐藏层）=====
model_single = SingleLayerPerceptron()  # 创建单层感知机
optimizer_single = torch.optim.Adam(model_single.parameters(), lr=0.5)  # Adam 优化器

losses_single = []  # 单层感知机损失记录
for epoch in range(500):
    optimizer_single.zero_grad()  # 清零梯度
    y_pred = model_single(X_xor)  # 前向传播
    loss = criterion(y_pred, y_xor)  # 计算损失
    losses_single.append(loss.item())  # 记录损失
    loss.backward()  # 反向传播
    optimizer_single.step()  # 更新参数

# ===== 创建网格 =====
x1_range = np.linspace(-0.5, 1.5, 100)  # X1 轴范围
x2_range = np.linspace(-0.5, 1.5, 100)  # X2 轴范围
X1, X2 = np.meshgrid(x1_range, x2_range)  # 生成网格
grid_points = torch.tensor(np.column_stack([X1.ravel(), X2.ravel()]), dtype=torch.float32)  # [10000, 2]

# 颜色标记数据点
point_colors = ['green' if y == 1 else 'red' for y in y_xor.numpy()]  # 1=绿，0=红

# ===== 画图 =====
plt.figure(figsize=(14, 5))

# 左图：损失曲线对比
plt.subplot(1, 2, 1)
plt.plot(losses_mlp, label='MLP (有隐藏层)', alpha=0.8, linewidth=2)  # MLP 损失
plt.plot(losses_single, label='单层感知机', alpha=0.8, linewidth=2)  # 单层损失
plt.xlabel('训练轮次')  # x 轴标签
plt.ylabel('损失')  # y 轴标签
plt.title('训练损失对比')  # 标题
plt.legend()  # 图例
plt.grid(True, alpha=0.3)  # 网格

# 右图：单层感知机的决策边界
model_single.eval()  # 评估模式
with torch.no_grad():  # 不计算梯度
    y_grid_single = model_single(grid_points).numpy().reshape(X1.shape)  # 预测网格

plt.subplot(1, 2, 2)
plt.contourf(X1, X2, y_grid_single, levels=20, cmap='RdYlGn', alpha=0.8)  # 热力图
plt.contour(X1, X2, y_grid_single, levels=[0.5], colors='black', linewidths=2)  # 决策边界
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=point_colors, s=300, edgecolors='black', linewidth=2)  # 数据点
plt.xlabel('X1')  # x 轴标签
plt.ylabel('X2')  # y 轴标签
plt.title('单层感知机的决策边界\n(永远是一条直线！)')  # 标题
plt.grid(True, alpha=0.3)  # 网格

plt.tight_layout()  # 自动调整布局
plt.savefig('day05_single_vs_mlp.png', dpi=150, bbox_inches='tight')  # 保存图片
plt.show()  # 显示图片

# ===== 打印对比结果 =====
print("\n对比结果：")
print("=" * 55)

print("单层感知机（无隐藏层）：")
model_single.eval()  # 评估模式
with torch.no_grad():  # 不计算梯度
    y_single = model_single(X_xor)  # 获取预测
for i in range(4):
    pred = 1 if y_single[i].item() > 0.5 else 0  # 预测值
    true = int(y_xor[i].item())  # 真实标签
    status = "✓ 正确" if pred == true else "✗ 错误"  # 对错
    print(f"  ({int(X_xor[i,0])}, {int(X_xor[i,1])}) → 预测={pred}, 真实={true} {status}")

print("\nMLP（有隐藏层）：")
model_mlp.eval()  # 评估模式
with torch.no_grad():  # 不计算梯度
    y_mlp = model_mlp(X_xor)  # 获取预测
for i in range(4):
    pred = 1 if y_mlp[i].item() > 0.5 else 0  # 预测值
    true = int(y_xor[i].item())  # 真实标签
    status = "✓ 正确" if pred == true else "✗ 错误"  # 对错
    print(f"  ({int(X_xor[i,0])}, {int(X_xor[i,1])}) → 预测={pred}, 真实={true} {status}")

print("\n结论：")
print("  单层感知机的决策边界永远是一条直线，无法分开 XOR 的对角线分布。")
print("  MLP 通过隐藏层实现非线性决策边界，成功解决 XOR！")

---
## 9. 翻译词典

| 生活直觉 | 深度学习术语 | 英文 |
|----------|-------------|------|
| 自动炒菜机，按下按钮就行 | 自动求导 | Automatic Differentiation (Autograd) |
| 「这个参数要学」 | 梯度追踪 | `requires_grad=True` |
| 机器自动算出每味料的影响 | 反向传播自动完成 | `loss.backward()` |
| 自动调味器 | 优化器 | Optimizer (Adam/SGD) |
| 调味器自己调整调料 | 参数更新 | `optimizer.step()` |
| 清空上一轮的调味记录 | 梯度清零 | `optimizer.zero_grad()` |
| 洗衣机的操作面板 | nn.Module | 模型基类 |
| 弯曲的分界线 | 非线性决策边界 | Non-linear Decision Boundary |

---
## 10. 第二周总结

这一周，我们从零构建了一个完整的神经网络：

| 天 | 主题 | 核心概念 | 关键突破 |
|----|------|----------|----------|
| Day01 | 隐藏层 | 空间变换 | 单层 → 多层，线性不可分 → 线性可分 |
| Day02 | 激活函数 | Sigmoid/ReLU | 让网络有非线性能力 |
| Day03 | 损失函数与梯度下降 | MSE/BCE, Adam | 衡量错误 + 知道怎么改 |
| Day04 | 反向传播 | 链式法则 | 把误差分摊给每个权重 |
| Day05 | 完整训练 | autograd + nn.Module | 从手写到自动化 |

### 从手写到自动化的进化

```
Day01-02: 手动定义权重，前向传播，没有训练
     ↓
Day03:   加入损失函数 + 梯度下降，开始「训练」
     ↓
Day04:   手写反向传播，理解链式法则
     ↓
Day05:   用 autograd 自动反向传播，用 nn.Module 封装模型
```

### 明天预告

第二周解决了「怎么训练一个全连接网络」的问题。但全连接网络处理图像有一个致命缺陷：**它不理解空间结构**。

一张 28×28 的图片被展平成 784 维向量后，像素之间的空间关系完全丢失了。

下周进入**第三周：视觉征服**。学习**卷积神经网络（CNN）**——让机器「看见」图片中的模式。